# Wedding planner

In [3]:
start_time=time_ns()
#*************************************************************************
# WeddingPlanner Assignment, "Mathematical Programming Modelling" (42112)
# utilizing Hamming-distance heurisitc optimization
#************************************************************************
# Intro definitions
using JuMP
using HiGHS
#using Gurobi
#************************************************************************

let
    #************************************************************************
    # PARAMETERS
    # Set: Guests, g,g1,g2
    # Set: Tables, t
    # DATA: SharedInterests[g1,g2]: 2-dim Guests*Guests: No of shared interests
    # DATA: Couple[g1,g2]: 2-dim Guests*Guests: 1 if two guests are partners, else 0 (symmetric)
    # DATA: Age[g]: 1-dim, Guests: Age measured in years
    # DATA: Male[g]: 1-dim, Guests: 1 if guest Male, 0 otherwise
    # DATA: Female[g]: 1-dim, Guests: 1 if guest Female, 0 otherwise
    # DATA: Know[g1,g2]: 2-dim Guests*Guests: 1 if two guests know each other (symmetric)
    time_limit=120
    K_initial=5 #"Neighbourhood size"
	K=K_initial 
    include("Data/WeddingData74.jl") # path to data
    G=length(Guests)
    T=length(Tables)
    TableCap=ceil(G/T)
    println("Runing Hamming-Distance-Heuristic on the WeddingPlanner with "*
			"$(G) guests, $(T) tables with capacity $(TableCap)\n")
    #************************************************************************
    
    #************************************************************************
    # Model
    wp =Model(HiGHS.Optimizer)
    #wp =Model(Gurobi.Optimizer)
    set_silent(wp) #To not get print from solver
    #Set timelimit pr MP solve to less than full timelimit
    set_time_limit_sec(wp, time_limit/2) 

    # 1 if guest g is sitting at table T
    @variable(wp, x[g=1:G,t=1:T], Bin)
	
	# Slack, if guest is NOT seated anywhere:
    @variable(wp, slack[g=1:G], Bin)

    # 1 if guest g1 and guest g2 are both sitting at table T
    @variable(wp, 0 <= y[g1=1:G,g2=1:G,t=1:T] <= ( g1 < g2 ? 1 : 0) ) 
    
    # Sum of different objectives
    @objective(wp, Max,
               # maximize total shared interests
               sum( SharedInterests[g1,g2]*y[g1,g2,t]
                    for t=1:T, g1=1:G, g2=1:G if g1<g2)
               
               # penalty of not planning all guests
               - sum( G*slack[g] for g=1:G) )                   
    
    # all guests have to sit at a table or have slack variable =1
    @constraint(wp, [g=1:G],
                slack[g] + sum( x[g,t] for t=1:T) == 1)
    
    # dont exceed the number of persons at a table
    @constraint(wp, [t=1:T],
                sum( x[g,t] for g=1:G) <= TableCap)
    
    # couples should sit at the same table
    @constraint(wp, [g1=1:G,g2=1:G,t=1:T; Couple[g1,g2]==1],
                x[g1,t] == x[g2,t])
    
    # limit y, can only become 1 if g1 is at the table t
    @constraint(wp, [g1=1:G,g2=1:G,t=1:T; g1<g2],
                y[g1,g2,t]  <= x[g1,t])
    
    # limit y, can only become 1 if g2 is (also) at the table t
    @constraint(wp, [g1=1:G,g2=1:G,t=1:T; g1<g2],
                y[g1,g2,t]  <= x[g2,t])
    #************************************************************************

    #************************************************************************
    
	#Function to add constraint that forces new solution and solves model again:
    function AddKConstraintAndOptimize(xVal,K)
        
        @constraint(wp, Kconstraint,
                    sum( x[g,t] for g=1:G,t=1:T if xVal[g,t]==0) +
                        sum( (1-x[g,t]) for g=1:G,t=1:T if xVal[g,t]==1)
                    <= K)
        optimize!(wp)
		
		#Extract infromation on the new solution:
        xVal=round.(Int,value.(x))
        slackVal=round.(Int,value.(slack))
        curObj=round(Int64,objective_value(wp))
		
		#Remove the newly added constraint from the model after use:
        delete(wp, Kconstraint)
        unregister(wp, :Kconstraint)
		
        return (curObj,xVal,slackVal)
    end

    println("Starting Hamming Distance Heuristic")
    xVal=zeros(Int8,G,T) #Start with empty solution, NO guests are seated

    it=1
    curObj=-1
    oldObj=-1
    
    while (time_ns()-start_time)/1.0e9<time_limit
        (curObj,xVal,slackVal)=AddKConstraintAndOptimize(xVal,K)
        this_time=(time_ns()-start_time)/1.0e9
        println("It: $(it) Time: $(round(this_time,digits=1)) K: $(K) "*
				"CurObj: $(curObj) Slack: $(sum(slackVal[g] for g=1:G))")
        if curObj<=oldObj # comparison for maximization problems
            K=K+1 #If no improvement, increase neighbourhood.
        else
            K=K_initial #If there IS improvement then search with small neighbourhood for as long as possible
        end
        oldObj=curObj
        it+=1
    end
    #************************************************************************
    
    #************************************************************************
    println("Successfull end of $(PROGRAM_FILE)")
    #************************************************************************
end


Runing Hamming-Distance-Heuristic on the WeddingPlanner with 74 guests, 9 tables with capacity 9.0

Starting Hamming Distance Heuristic
It: 1 Time: 63.5 K: 5 CurObj: -5103 Slack: 69
It: 2 Time: 123.6 K: 6 CurObj: -4602 Slack: 63
Successfull end of 


k = 1 -> It: 127 Time: 145.5 K: 8 CurObj: 311 Slack: 0

I assume that when we increasse k we will get a better solution, since we are allowed to change more variables

k = 5 -> It: 2 Time: 123.6 K: 6 CurObj: -4602 Slack: 63

Well, it didn't get a change to try many solutions so it stops after two iterations with just Total - 63 guest seated
